# Self-supervised learning of person characteristics

**Sensori** maps a day of wrist movement to an embedding (a vector of 768 numbers) learned through self-supervision. This practical explores how these embeddings capture person characteristics and support health-outcome prediction.

The **National Health and Nutrition Examination Survey (NHANES)** assesses health and nutrition in the United States through interviews and physical examinations. We use wrist accelerometer data from the [2011–2012](https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2011/DataFiles/PAX80_G.htm) and [2013–2014](https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/PAX80_H.htm) surveys, which asked participants to wear a monitor for seven consecutive days.

The practical covers:

1. Extracting embeddings for 10 example participants and comparing them with saved results.
2. Exploring similarities between participants and associations with their characteristics.
3. Evaluating health-outcome prediction using embeddings and demographic variables.

Check the device and file paths below, and run the cells in order.


In [ ]:
import gc
import json
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

from model import load_sensori

SEED = 111
np.random.seed(SEED)
torch.manual_seed(SEED)

## 1. Generate embeddings for 10 example participants

Only days with at least **22 hours of valid wear time** were retained. The 10 example participants therefore have different numbers of valid days.

Each file contains 24 hours of triaxial acceleration at 10 Hz, measured in g. The array shape is `(2880, 300, 3)`: 2,880 consecutive 30-second windows, with 300 samples and three axes per window.


In [ ]:
DEVICE = 0  # GPU number (0, 1, ...) or 'cpu'.
device = torch.device('cpu' if DEVICE == 'cpu' else f'cuda:{DEVICE}')
if device.type == 'cpu':
    torch.set_num_threads(4)
print(f'Using {device}')

DATA_DIR = Path('/srv/sensori')


In [ ]:
# Load the full-cohort reference embeddings and list the 10 example participants.
reference_embeddings = np.load(DATA_DIR / 'nhanes_embeddings.npy', allow_pickle=True).item()
example_ids = sorted(path.name for path in (DATA_DIR / 'nhanes_days').iterdir() if path.is_dir())

# Keep each person's files in numeric day order: day_0, day_1, day_2, ...
day_files = {}
for eid in example_ids:
    day_files[eid] = sorted(
        (DATA_DIR / 'nhanes_days' / eid).glob('day_*.npy'),
        key=lambda path: int(path.stem.split('_')[1]),
    )

example_day = np.load(day_files[example_ids[0]][0])
print('Saved day:', example_day.shape)
del example_day

for eid in example_ids:
    print(f'Example participant {eid}: {len(day_files[eid])} valid days')

### Inspect the acceleration signal

Plot the preprocessed x, y and z acceleration at 10 Hz for one participant, with one row per day.


In [ ]:
example_days = day_files[example_ids[0]]
fig, axes = plt.subplots(
    len(example_days), 1, figsize=(7, len(example_days)),
    sharex=True, sharey=True, squeeze=False, constrained_layout=True,
)
for axis, day_path in zip(axes.flat, example_days):
    xyz = np.load(day_path).reshape(-1, 3)
    hours = np.arange(len(xyz)) / (10 * 3600)
    axis.plot(hours, xyz, linewidth=0.5)
    axis.set_ylabel(day_path.stem.replace('_', ' '))

axes[0, 0].legend(['x', 'y', 'z'], loc='upper right', ncol=3)
axes[-1, 0].set(xlabel='Hour of day', xlim=(0, 24))
fig.supylabel('Acceleration (g)')
plt.show()

# Release the raw samples and the figure's stored line data after display.
plt.close(fig)
del fig, axes, axis, xyz, hours
_ = gc.collect()


### Load the pretrained model

Use the pretrained Sensori model in evaluation mode to extract one embedding per day. The model weights remain fixed throughout extraction.

To match Sensori's input format, reshape each day into one-minute windows with shape `(1, 1440, 600, 3)` before extracting embeddings.


In [ ]:
torch.set_float32_matmul_precision('high')
config = SimpleNamespace(**json.loads((DATA_DIR / 'config_model.json').read_text()))
model = load_sensori(DATA_DIR / 'model.pt', config, device=device)
model.eval()
model.requires_grad_(False)

live_embeddings = {}
with torch.inference_mode():
    for eid in tqdm(example_ids, desc='Participants'):
        person_embeddings = []
        for day_path in day_files[eid]:
            day = np.load(day_path)
            x = torch.from_numpy(day.reshape(1, 1440, 600, 3)).to(device)
            person_embeddings.append(model(x).squeeze(0).cpu().numpy())
        live_embeddings[eid] = np.stack(person_embeddings)

# Release the model and raw day arrays after extraction.
del model, day, x, person_embeddings
_ = gc.collect()
if device.type == 'cuda':
    torch.cuda.empty_cache()

Compare the extracted and precomputed embeddings using cosine similarity.


In [ ]:
live = np.concatenate([live_embeddings[eid] for eid in example_ids])
reference = np.concatenate([reference_embeddings[eid] for eid in example_ids])
cosine = np.sum(live * reference, axis=1) / (
    np.linalg.norm(live, axis=1) * np.linalg.norm(reference, axis=1)
)
print(f'Minimum cosine similarity: {cosine.min():.6f}')


### Inspect the similarity between person-days

Each row and column represents a newly extracted embedding for one day. Days are ordered by participant, then day; white lines separate participants. Similarities of 0.8 or lower share the darkest colour. The blank diagonal excludes each day's comparison with itself.

Cosine similarity measures agreement in direction: values closer to 1 indicate more similar embeddings. Are days from the same person more similar than days from different people?


In [ ]:
days = [live_embeddings[eid] for eid in example_ids]
X = np.concatenate(days)
X = X / np.linalg.norm(X, axis=1, keepdims=True)
similarity = X @ X.T
np.fill_diagonal(similarity, np.nan)  # Hide each day's comparison with itself.

# Mark the boundaries and centres of each participant's days.
edges = np.cumsum([0] + [len(person_days) for person_days in days])
centres = (edges[:-1] + edges[1:] - 1) / 2
labels = [f'P{i + 1}' for i in range(len(example_ids))]

fig, axis = plt.subplots(figsize=(6, 5), constrained_layout=True)
image = axis.imshow(similarity, cmap='magma', vmax=1, vmin=0.8)
for boundary in edges[1:-1] - 0.5:
    axis.axhline(boundary, color='white', linewidth=0.7)
    axis.axvline(boundary, color='white', linewidth=0.7)
axis.set(
    xticks=centres, xticklabels=labels, yticks=centres, yticklabels=labels,
    xlabel='Days grouped by participant', ylabel='Days grouped by participant',
    title='Similarity of individual day embeddings',
)
colorbar = fig.colorbar(image, ax=axis, label='Cosine similarity', shrink=0.8)
colorbar.set_ticks([0.8, 0.85, 0.9, 0.95, 1], labels=['≤0.8', '0.85', '0.9', '0.95', '1'])
plt.show()


## 2. Explore person characteristics

From here on, use the precomputed embeddings for the full cohort.


### Participant information

`nhanes_tabular.csv` contains demographics, examination measurements and questionnaire responses. Participant IDs (`eid`) link these records to the embeddings.

In [ ]:
tabular = pd.read_csv('nhanes_tabular.csv')
tabular.head()

### Participant embeddings

L2-normalize each daily embedding to unit length, then average across days to obtain one embedding per participant. Include participants with both embeddings and tabular data.

In [ ]:
participant_embeddings = {}
for eid, days in reference_embeddings.items():
    days = np.asarray(days, dtype=np.float32)
    norms = np.linalg.norm(days, axis=1)
    valid = np.isfinite(norms) & (norms > 0)
    if valid.any():
        participant_embeddings[str(eid)] = (days[valid] / norms[valid, None]).mean(axis=0)

embeddings = pd.DataFrame.from_dict(participant_embeddings, orient='index')
embeddings.index.name = 'eid'
embedding_columns = [f'embedding_{i:03d}' for i in range(embeddings.shape[1])]
embeddings.columns = embedding_columns

tabular['eid'] = tabular['eid'].astype(str)
cohort = tabular.set_index('eid').join(embeddings, how='inner')
# Keep the participant table; the full daily embedding arrays are no longer needed.
del reference_embeddings, participant_embeddings, embeddings, days, norms, valid
_ = gc.collect()
print(f'Participants with tabular data and embeddings: {len(cohort):,}')
cohort.reset_index()[['eid', 'cycle', 'age', 'Sex', 'BMI']].head()


### Visualize the embeddings with PCA

**Principal component analysis (PCA)** projects the embeddings onto two linear directions that capture the greatest variation. Each point represents a participant; age, BMI, sex and device firmware are used only to colour the resulting map.

Compare the patterns associated with demographics and firmware. The accompanying table shows firmware versions by survey cycle.

In [ ]:
pca_table = cohort.copy()
pca_table[['PC1', 'PC2']] = PCA(n_components=2, random_state=SEED).fit_transform(
    pca_table[embedding_columns]
)


def version_key(value):
    return tuple(map(int, value.split('.')))


firmware_versions = sorted(pca_table['firmware'].unique(), key=version_key)
early = [version for version in firmware_versions if version_key(version) < (2, 3)]
late = [version for version in firmware_versions if version_key(version) >= (2, 3)]
firmware_colors = {
    **dict(zip(early, plt.get_cmap('Blues')(np.linspace(0.45, 0.9, len(early))))),
    **dict(zip(late, plt.get_cmap('Oranges')(np.linspace(0.45, 0.9, len(late))))),
}

fig, axes = plt.subplots(2, 2, figsize=(12, 9), constrained_layout=True)
axes = axes.ravel()
for axis, column, title, cmap, upper in (
    (axes[0], 'age', 'Age (years)', 'viridis', 80),
    (axes[1], 'BMI', 'BMI (kg/m²)', 'magma', 40),
):
    table = pca_table.dropna(subset=[column])
    values = table[column].clip(upper=upper)
    points = axis.scatter(table['PC1'], table['PC2'], c=values, s=5, cmap=cmap)
    colorbar = fig.colorbar(points, ax=axis, label=title)
    ticks = [tick for tick in colorbar.get_ticks() if values.min() <= tick < upper]
    colorbar.set_ticks([*ticks, upper], labels=[*(f'{tick:g}' for tick in ticks), f'≥{upper}'])
    axis.set_title(title)

for axis, column, title, colors in (
    (axes[2], 'Sex', 'Sex', {'Female': '#D95F82', 'Male': '#2878B5'}),
    (axes[3], 'firmware', 'Firmware version', firmware_colors),
):
    table = pca_table.dropna(subset=[column]).sample(frac=1, random_state=SEED)
    axis.scatter(table['PC1'], table['PC2'], c=table[column].map(colors), s=5)
    for value, color in colors.items():
        axis.scatter([], [], color=color, label=value)
    axis.set_title(title)
    axis.legend(ncol=2 if column == 'firmware' else 1)

for axis in axes:
    axis.set(xticks=[], yticks=[], xlabel='PC 1', ylabel='PC 2')
plt.show()

firmware_distribution = pd.crosstab(
    pca_table['firmware'], pca_table['cycle'], margins=True, margins_name='Total'
).reindex([*firmware_versions, 'Total'])
firmware_distribution.index.name = 'Firmware'
firmware_distribution

## 3. Predicting health targets

The analysis includes regular alcohol consumption, current tobacco smoking, overall health rating and 19 physical-function outcomes. Responses are mapped to binary labels, with excluded responses treated as missing.

Label 1 means regular drinking, current smoking, good/very good/excellent health, or at least some difficulty with the specified activity.

We include participants aged **under 80**. NHANES [records ages 80 and above as 80](https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2011/DataFiles/DEMO_G.htm#RIDAGEYR), so we exclude these participants because their exact ages are unavailable for the age, sex and BMI comparison. The summary reports the available sample size and proportion labelled 1 for each target; consult the mappings for label definitions.

In [ ]:
with (DATA_DIR / 'covariate_mappings.json').open(encoding='utf-8') as handle:
    mappings = json.load(handle)

TARGETS = [name for name in mappings['categorical_mappings'] if name != 'Sex']
model_data = cohort[cohort['age'] < 80].copy()
print(f'Participants younger than 80: {len(model_data):,}')

for column in ['Sex', *TARGETS]:
    mapping = mappings['categorical_mappings'][column]
    if '$ref' in mapping:
        mapping = mappings['shared_mappings'][mapping['$ref']]
    unexpected = set(model_data[column].dropna().unique()) - set(mapping)
    if unexpected:
        raise ValueError(f'Unmapped values for {column}: {sorted(unexpected, key=str)}')
    model_data[column] = pd.to_numeric(model_data[column].map(mapping), errors='coerce')

target_summary = pd.DataFrame({
    'participants': model_data[TARGETS].notna().sum(),
    'positive_ratio': model_data[TARGETS].mean(),
})
target_summary.round(3)


### Linear probes

A **linear probe** tests whether a self-supervised or foundation model's embeddings are useful for a labelled task. We keep Sensori fixed and train a simple predictor on its embeddings. This provides a quick way to assess the learned representation—for example, whether it captures information about difficulty walking.

We use **logistic regression** to compare two feature sets:

1. Age, sex and BMI.
2. Sensori embeddings.

We use a fixed **70:30 training/test split by participant**, shared across all targets and both feature sets. For each feature set, imputation and scaling are fitted once on the training participants and applied to the test participants. Each predictor uses only participants with an available label for that target.

**AUROC (area under the receiver operating characteristic curve)** measures how well predictions rank positive examples above negative examples: 0.5 corresponds to chance ranking and 1.0 to perfect ranking.

**Regularization** discourages large coefficients. We use `C=1` for demographics and `C=0.001` for embeddings; smaller `C` means stronger regularization. Logistic regression predicts $\hat p_i$ and learns its weights by minimizing the [L2-regularized loss](https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression):

$$
\hat p_i = \frac{1}{1 + \exp[-(\mathbf{w}^{\mathsf T}\mathbf{z}_i + b)]},
$$

$$
\mathcal{L}(\mathbf{w}, b)
= -\sum_{i=1}^{n} a_i\left[y_i\log\hat p_i + (1-y_i)\log(1-\hat p_i)\right]
+ \frac{\|\mathbf{w}\|_2^2}{2C}.
$$

Here, $\mathbf{z}_i$ is the participant's feature vector after imputation and scaling, $y_i$ is their binary label, and $\hat p_i$ is the predicted probability of label 1. We learn the coefficients $\mathbf{w}$ and intercept $b$. The weights $a_i$ balance the two classes (`class_weight='balanced'`).


In [ ]:
train_data, test_data = train_test_split(model_data, test_size=0.3, random_state=SEED)
print(f'Training participants: {len(train_data):,}; test participants: {len(test_data):,}')


def evaluate_target(X_train, X_test, y_train, y_test, C):
    train_mask = y_train.notna()
    test_mask = y_test.notna()
    model = LogisticRegression(C=C, max_iter=10_000, class_weight='balanced')
    model.fit(X_train[train_mask], y_train[train_mask].astype(int))
    scores = model.predict_proba(X_test[test_mask])[:, 1]
    return roc_auc_score(y_test[test_mask].astype(int), scores)


FEATURE_SETS = {
    'Age + sex + BMI': (['age', 'Sex', 'BMI'], 1.0),
    'Sensori embedding': (embedding_columns, 0.001),
}

results = []
for feature_set, (columns, C) in FEATURE_SETS.items():
    imputer = SimpleImputer(strategy='median')
    X_train = imputer.fit_transform(train_data[columns])
    X_test = imputer.transform(test_data[columns])

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    for target in tqdm(TARGETS, desc=feature_set):
        auroc = evaluate_target(X_train, X_test, train_data[target], test_data[target], C)
        results.append({'target': target, 'feature_set': feature_set, 'AUROC': auroc})


### Prediction performance

Each bar shows AUROC on the held-out test participants for one target and feature set.


In [ ]:
results = pd.DataFrame(results)
summary = results.pivot(index='target', columns='feature_set', values='AUROC')

COLORS = ('#6B7280', '#E66101')
fig, axis = plt.subplots(figsize=(6, 8))
positions = np.arange(len(TARGETS))
for offset, feature_set, color in zip((-0.18, 0.18), FEATURE_SETS, COLORS):
    axis.barh(
        positions + offset, summary.loc[TARGETS, feature_set], height=0.34,
        color=color, label=feature_set,
    )
axis.axvline(0.5, color='black', linestyle='--', linewidth=0.8, alpha=0.5)
axis.set(
    yticks=positions, yticklabels=[target.replace('_', ' ') for target in TARGETS],
    xlabel='Test AUROC', xlim=(0.5, 1),
)
axis.tick_params(axis='y', labelsize=9, length=0)
axis.invert_yaxis()
axis.spines[['top', 'right']].set_visible(False)
axis.legend(frameon=False, loc='lower left', bbox_to_anchor=(0, 1), ncol=2, fontsize=9)
plt.show()


## Bonus: Add daily step count to age, sex and BMI

`nhanes_device_features.csv` contains six physical-activity measures and five step measures from the NHANES wrist recordings. Here, we add mean daily steps to age, sex and BMI.


In [ ]:
device_data = pd.read_csv('nhanes_device_features.csv', dtype={'eid': str}).set_index('eid')
train_with_steps = train_data.join(device_data)
test_with_steps = test_data.join(device_data)

columns = ['age', 'Sex', 'BMI', 'StepsDayAvgAdjusted']
imputer = SimpleImputer(strategy='median')
X_train_steps = imputer.fit_transform(train_with_steps[columns])
X_test_steps = imputer.transform(test_with_steps[columns])

scaler = StandardScaler()
X_train_steps = scaler.fit_transform(X_train_steps)
X_test_steps = scaler.transform(X_test_steps)

step_scores = []
for target in tqdm(TARGETS, desc='Age + sex + BMI + steps'):
    step_scores.append(evaluate_target(
        X_train_steps, X_test_steps,
        train_with_steps[target], test_with_steps[target], C=1.0,
    ))

step_comparison = pd.DataFrame({
    'Age + sex + BMI': summary.loc[TARGETS, 'Age + sex + BMI'],
    'Age + sex + BMI + steps': step_scores,
    'Sensori embedding': summary.loc[TARGETS, 'Sensori embedding'],
})


In [ ]:
fig, axis = plt.subplots(figsize=(6, 8))
positions = np.arange(len(TARGETS))
for offset, column, color in zip((-0.25, 0, 0.25), step_comparison, ('#6B7280', '#00796B', '#E66101')):
    axis.barh(
        positions + offset, step_comparison[column], height=0.23,
        color=color, label=column,
    )
axis.set(
    yticks=positions, yticklabels=[target.replace('_', ' ') for target in TARGETS],
    xlabel='Test AUROC', xlim=(0.5, 1),
)
axis.tick_params(axis='y', labelsize=9, length=0)
axis.invert_yaxis()
axis.spines[['top', 'right']].set_visible(False)
axis.legend(frameon=False, loc='lower left', bbox_to_anchor=(0, 1), ncol=1, fontsize=9)
plt.show()


### Exercise: Add more device features and vary regularization

Copy the two cells above and try the following:

1. Add all 11 device features to age, sex and BMI:
   ```python
   columns = ['age', 'Sex', 'BMI'] + device_data.columns.tolist()
   ```
2. Try `C=0.001`, `0.01`, `0.1`, `1` and `10` in `evaluate_target`. Smaller `C` means stronger regularization. Keep the same training/test split and rerun preprocessing when changing the features.
3. Update the plot label to **Age + sex + BMI + device features**. How does AUROC change across targets and values of `C`? How does it compare with Sensori embeddings?



### Further reading and exploration

- **Paper:** [Learning Human Health and Diseases from 24-hour Wrist Movement](https://arxiv.org/abs/2608.29494)
- **Code:** [Sensori GitHub repository](https://github.com/OxWearables/Sensori)
